# Darkroom — run everything on a free Colab GPU

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`.

Run the cells **top to bottom** with the ▶ button (or `Shift+Enter`). Each cell says what it does and how long it takes.
Edit only the cell marked **EDIT ME**.

## 1. Check the GPU  ·  ~5 s
You should see a Tesla T4 (or better). If you see an error, the runtime is not set to GPU — see the note above.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. EDIT ME — your team details

In [ ]:
TEAM_ID = "T03"                                   # your team ID from the organisers
USNS    = "1RUA24CSE0305,1RUA24CSE247,1RUA24CSE0280,1RUA24CSE0253,1RUA24CSE0215"      # every member's USN, comma-separated, no spaces
QUICK   = True                                    # True ≈ 30-45 min on a T4; False = full run (≈2 h on a T4)
APP_URL = "https://YOUR-APP.lovable.app"          # your published Lovable link (used to build the judges' link)

## 3. Get the code and install  ·  ~3 min

In [ ]:
%cd /content
import os
if os.path.isdir("prachi-darkroom/.git"):
    !cd prachi-darkroom && git pull -q          # keeps already-downloaded data
else:
    !git clone -q https://github.com/PrachiAgarwal-coder/prachi-darkroom.git
%cd /content/prachi-darkroom/server
!pip install -q -r requirements.txt
# Colab ships an old torchao that makes peft refuse to load LoRAs; we don't use torchao.
!pip uninstall -y -q torchao
print("Installed.")

## 4. Download the evaluation data (MS-COCO 2017 val)  ·  ~3-5 min
Downloads the official caption file once (~240 MB), then 300 images. Nothing to do by hand.

In [ ]:
!python ../eval/download_coco.py --n 300 --out ../data/coco

## 5. Run the evaluation  ·  QUICK ≈ 30-45 min, full ≈ 2 h
The first run also downloads the models (~5 GB). It prints progress tier by tier. **Keep this tab open.**

In [ ]:
args = "--n 100 --n-baseline 50 --n-ablation 20" if QUICK else ""
!python ../eval/evaluate.py --team "$TEAM_ID" --usns "$USNS" --coco ../data/coco $args

## 5b. Measure memory per tier, in isolation  ·  ~10 min
Peak memory in step 5 includes every tier plus the scorer models in one process. This loads **one tier per process** and replaces those numbers with real per-tier peaks, then rewrites the report.
Works on a fresh session too: it reads `eval/results/results.json` from GitHub, so you only need cells 1–3 first.

In [ ]:
!pkill -f uvicorn; pkill -f cloudflared; sleep 2      # free the GPU if the live server was started
!python ../eval/measure_memory.py --team "$TEAM_ID" --usns "$USNS"

## 6. Look at the results

In [ ]:
import os
from IPython.display import Markdown, Image, display
report = f"../eval/results/eval_P3_{TEAM_ID}.md"
if not os.path.exists(report):
    print("No results yet - the evaluation cell above did not finish. Scroll up for the error.")
else:
    display(Image("../eval/results/quality_vs_cost.png"))
    display(Markdown(open(report).read()))

## 7. Download the results  ·  then upload them to GitHub
This downloads `eval_results.zip`. Unzip it on your laptop and upload the **contents** into the repo's `eval/results/` folder on GitHub (Add file → Upload files).

In [ ]:
!cd ../eval && zip -qr /content/eval_results.zip results -x "results/images/*"
from google.colab import files
files.download("/content/eval_results.zip")

## 8. Meaningful adaptation — train your own LoRA  ·  ~30 min total
Fine-tunes a small LoRA on COCO **train2017**, which never overlaps the val2017 images used for evaluation, then evaluates the Balanced tier with it using the same 100 captions and seeds as step 5, so the numbers are directly comparable.
Needs cells 1–4 first (cell 4 provides the evaluation images). Run these three cells in order; don't start the live server while they run.

In [ ]:
!python ../eval/download_coco.py --split train --n 600 --out ../data/coco_train        # COCO train2017, disjoint from the val2017 eval set, ~3 min
!python ../eval/train_lora.py --data ../data/coco_train --steps 400 --out ../lora/coco     # ~20 min on a T4, prints train/val loss
import os; assert os.path.isfile("../lora/coco/pytorch_lora_weights.safetensors"), "LoRA was not saved; read the error above"
print("LoRA saved:", os.listdir("../lora/coco"))

In [ ]:
import os
os.environ["DARKROOM_CUSTOM_LORA"] = "/content/prachi-darkroom/lora/coco"
!python ../eval/evaluate.py --team "$TEAM_ID" --usns "$USNS" --coco ../data/coco --tiers balanced --n 100 --skip-ablation --out ../eval/results_lora --no-root-copy
del os.environ["DARKROOM_CUSTOM_LORA"]
import json
base = json.load(open("../eval/results/results.json"))["tiers"]["balanced"]
lora = json.load(open("../eval/results_lora/results.json"))["tiers"]["balanced"]
for k in ["clip", "kid", "fid", "object_recall", "relation_acc", "prompt_pass", "gen_ms_mean"]:
    print(f"{k:15s} without LoRA {base.get(k, float('nan')):9.4f}   with LoRA {lora.get(k, float('nan')):9.4f}")

In [ ]:
!cd .. && zip -qr /content/lora_results.zip eval/results_lora -x "eval/results_lora/images/*"
from google.colab import files
files.download("/content/lora_results.zip")      # upload its eval/results_lora folder to GitHub too

---
## Optional A — live demo server (use this for judging)
Starts the model server on this GPU and opens a public `https://….trycloudflare.com` tunnel.
It stays alive **only while this tab is open and the laptop is awake**.

**Morning of judging (fresh Colab session):** set the T4 runtime, then run cells **1, 2 and 3** (skip 4–7) and then this cell. Total ≈ 10–15 min.
The cell prints a **link for judges**. Open it on your phone to check, then turn it into a QR code.

In [ ]:
import os, re, subprocess, time, urllib.request
os.environ.pop("DARKROOM_MOCK", None)
subprocess.Popen("cd /content/prachi-darkroom/server && nohup uvicorn darkroom.server:app --host 0.0.0.0 --port 8000 > /content/server.log 2>&1 &", shell=True)
if not os.path.exists("/content/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
subprocess.Popen("nohup /content/cloudflared tunnel --url http://localhost:8000 > /content/tunnel.log 2>&1 &", shell=True)

public = None
for _ in range(60):
    time.sleep(3)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read() if os.path.exists("/content/tunnel.log") else "")
    if m: public = m.group(0); break
print("Tunnel:", public or "not ready - rerun this cell")

print("Loading models (first time ~5-8 min)...", end="", flush=True)
for i in range(240):
    try:
        if '"ok":true' in urllib.request.urlopen("http://localhost:8000/health", timeout=3).read().decode():
            break
    except Exception:
        pass
    time.sleep(5); print(".", end="", flush=True)
print(" ready." if i < 239 else " still loading - check the next cell")

if public:
    ws = public.replace("https://", "wss://") + "/ws"
    print("\nModel server (also has a simple built-in UI):", public)
    print("WebSocket URL:", ws)
    print("\n>>> LINK FOR JUDGES:", f"{APP_URL.rstrip('/')}/?server={ws}")

In [ ]:
!tail -5 /content/server.log

## Optional B — deploy the permanent server to Modal from here
Run, open the link it prints, log in, then run the deploy cell. The URL it prints is your permanent model server.

In [ ]:
!pip install -q modal && modal token new

In [ ]:
%cd /content/prachi-darkroom
!modal deploy deploy/modal_app.py